# services.management

> Service layer wrapping graph plugin operations for document management

In [ ]:
#| default_exp services.management

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import Optional, List, Set, Dict, Any
import asyncio
from datetime import datetime, timezone

from fastcore.basics import patch

from cjm_plugin_system.core.manager import PluginManager
from cjm_graph_plugin_system.core import GraphContext, GraphNode, GraphEdge
from cjm_graph_domains.domains.relations import StructureRelations

from cjm_transcript_workflow_management.models import (
    DocumentSummary, DocumentDetail, SegmentSample,
    ExportBundle, ImportResult
)
from cjm_transcript_workflow_management.utils import truncate_text

## Debug Flag

In [ ]:
#| export
DEBUG_MANAGEMENT_SERVICE = False  # Enable for verbose graph query logging

## ManagementService

Core class with constructor, availability check, and all private helpers.
Public methods are added via `@patch` in subsequent cells.

In [ ]:
#| export
class ManagementService:
    """Service wrapping graph plugin operations for document management."""
    
    EXPORT_FORMAT = "cjm-context-graph"
    EXPORT_VERSION = "1.0.0"
    
    def __init__(
        self,
        plugin_manager: PluginManager,  # Plugin manager for accessing graph plugin
        plugin_name: str = "cjm-graph-plugin-sqlite",  # Name of the graph plugin
    ):
        """Initialize with plugin manager."""
        self._manager = plugin_manager
        self._plugin_name = plugin_name
    
    def is_available(self) -> bool:  # True if plugin is loaded and ready
        """Check if the graph plugin is available."""
        return self._manager.get_plugin(self._plugin_name) is not None
    
    # --- Plugin action wrappers ---
    
    async def _get_context_async(
        self,
        node_id: str,  # UUID of the node to query
        depth: int = 1,  # Traversal depth
    ) -> Optional[GraphContext]:  # GraphContext or None if error
        """Get node context from graph plugin."""
        try:
            result = await self._manager.execute_plugin_async(
                self._plugin_name,
                action="get_context",
                node_id=node_id,
                depth=depth,
            )
            return GraphContext.from_dict(result)
        except Exception as e:
            if DEBUG_MANAGEMENT_SERVICE:
                print(f"[ManagementService] get_context error: {e}")
            return None
    
    async def _find_nodes_by_label_async(
        self,
        label: str,  # Node label to search for
        limit: int = 1000,  # Maximum nodes to return
    ) -> Optional[Dict[str, Any]]:  # Raw result dict or None if error
        """Find nodes by label from graph plugin."""
        try:
            return await self._manager.execute_plugin_async(
                self._plugin_name,
                action="find_nodes_by_label",
                label=label,
                limit=limit,
            )
        except Exception as e:
            if DEBUG_MANAGEMENT_SERVICE:
                print(f"[ManagementService] find_nodes_by_label error: {e}")
            return None
    
    async def _delete_nodes_async(
        self,
        node_ids: List[str],  # UUIDs of nodes to delete
        cascade: bool = True,  # Also delete connected nodes and edges
    ) -> Optional[Dict[str, Any]]:  # Raw result dict or None if error
        """Delete nodes from graph plugin."""
        try:
            return await self._manager.execute_plugin_async(
                self._plugin_name,
                action="delete_nodes",
                node_ids=node_ids,
                cascade=cascade,
            )
        except Exception as e:
            if DEBUG_MANAGEMENT_SERVICE:
                print(f"[ManagementService] delete_nodes error: {e}")
            return None
    
    async def _export_graph_async(self) -> Optional[GraphContext]:  # Full graph or None if error
        """Export entire graph from plugin."""
        try:
            result = await self._manager.execute_plugin_async(
                self._plugin_name,
                action="export_graph",
            )
            return GraphContext.from_dict(result)
        except Exception as e:
            if DEBUG_MANAGEMENT_SERVICE:
                print(f"[ManagementService] export_graph error: {e}")
            return None
    
    async def _import_graph_async(
        self,
        graph_data: Dict[str, Any],  # Graph data dict with nodes and edges
        merge_strategy: str = "skip",  # How to handle ID collisions
    ) -> Optional[Dict[str, Any]]:  # Raw result dict or None if error
        """Import graph data into plugin."""
        try:
            return await self._manager.execute_plugin_async(
                self._plugin_name,
                action="import_graph",
                graph_data=graph_data,
                merge_strategy=merge_strategy,
            )
        except Exception as e:
            if DEBUG_MANAGEMENT_SERVICE:
                print(f"[ManagementService] import_graph error: {e}")
            return None
    
    # --- Extraction helpers ---
    
    def _find_document_node(
        self,
        context: GraphContext,  # Graph context from query
        document_id: str,  # Expected document UUID
    ) -> Optional[GraphNode]:  # Document node or None
        """Find the Document node in context."""
        for node in context.nodes:
            if node.label == 'Document' and node.id == document_id:
                return node
        return None
    
    def _get_segment_nodes(
        self,
        context: GraphContext,  # Graph context from query
    ) -> List[GraphNode]:  # Segment nodes sorted by index
        """Extract and sort Segment nodes from context."""
        segments = [n for n in context.nodes if n.label == 'Segment']
        segments.sort(key=lambda n: n.properties.get('index', 0))
        return segments
    
    def _count_edges_by_type(
        self,
        edges: List[GraphEdge],  # Edges from graph context
    ) -> Dict[str, int]:  # Counts keyed by relation type
        """Count edges by relation type."""
        counts = {'STARTS_WITH': 0, 'NEXT': 0, 'PART_OF': 0}
        for edge in edges:
            if edge.relation_type == StructureRelations.STARTS_WITH:
                counts['STARTS_WITH'] += 1
            elif edge.relation_type == StructureRelations.NEXT:
                counts['NEXT'] += 1
            elif edge.relation_type == StructureRelations.PART_OF:
                counts['PART_OF'] += 1
        return counts
    
    def _compute_timing_stats(
        self,
        segment_nodes: List[GraphNode],  # Sorted segment nodes
    ) -> Dict[str, Any]:  # total_duration, avg_duration, missing_count
        """Compute timing statistics from segment nodes."""
        total_duration = 0.0
        missing_count = 0
        valid_count = 0
        for node in segment_nodes:
            start = node.properties.get('start_time')
            end = node.properties.get('end_time')
            if start is not None and end is not None:
                total_duration += (end - start)
                valid_count += 1
            else:
                missing_count += 1
        avg_duration = total_duration / valid_count if valid_count > 0 else 0.0
        return {
            'total_duration': total_duration,
            'avg_duration': avg_duration,
            'missing_count': missing_count,
        }
    
    def _compute_source_stats(
        self,
        segment_nodes: List[GraphNode],  # Segment nodes
    ) -> Dict[str, Any]:  # missing_count, plugins list
        """Compute source statistics from segment nodes."""
        missing_count = 0
        plugins: Set[str] = set()
        for node in segment_nodes:
            sources = node.sources
            if not sources:
                missing_count += 1
            else:
                for source in sources:
                    if hasattr(source, 'plugin_name'):
                        plugins.add(source.plugin_name)
                    elif isinstance(source, dict) and 'plugin_name' in source:
                        plugins.add(source['plugin_name'])
        return {
            'missing_count': missing_count,
            'plugins': sorted(list(plugins)),
        }
    
    def _extract_samples(
        self,
        nodes: List[GraphNode],  # Segment nodes to convert
    ) -> List[SegmentSample]:  # Sample list
        """Convert segment nodes to sample objects."""
        return [self._node_to_sample(n) for n in nodes]
    
    def _node_to_sample(
        self,
        node: GraphNode,  # Segment node to convert
    ) -> SegmentSample:  # Sample object
        """Convert a single segment node to a SegmentSample."""
        return SegmentSample(
            index=node.properties.get('index', 0),
            text=truncate_text(node.properties.get('text', '')),
            start_time=node.properties.get('start_time', 0.0),
            end_time=node.properties.get('end_time', 0.0),
        )
    
    def _context_to_bundle(
        self,
        context: GraphContext,  # Graph context to wrap
        document_count: int,  # Number of Document nodes
    ) -> ExportBundle:  # Wrapped export bundle
        """Wrap a GraphContext in an ExportBundle with metadata."""
        return ExportBundle(
            format=self.EXPORT_FORMAT,
            version=self.EXPORT_VERSION,
            exported_at=datetime.now(timezone.utc).isoformat(),
            source_plugin=self._plugin_name,
            document_count=document_count,
            graph=context.to_dict(),
        )

### List Documents

In [ ]:
#| export
@patch
async def list_documents_async(self:ManagementService) -> List[DocumentSummary]:  # All documents sorted newest first
    """List all documents with summary info."""
    if DEBUG_MANAGEMENT_SERVICE:
        print("[ManagementService] list_documents called")
    
    if not self.is_available():
        return []
    
    # Find all Document nodes
    result = await self._find_nodes_by_label_async("Document")
    if result is None:
        return []
    
    # Convert raw node dicts to GraphNode objects
    raw_nodes = result.get("nodes", [])
    if DEBUG_MANAGEMENT_SERVICE:
        print(f"[ManagementService] Found {len(raw_nodes)} document nodes")
    
    summaries = []
    for node_dict in raw_nodes:
        node = GraphNode(
            id=node_dict["id"],
            label=node_dict.get("label", "Document"),
            properties=node_dict.get("properties", {}),
            sources=node_dict.get("sources", []),
            created_at=node_dict.get("created_at"),
            updated_at=node_dict.get("updated_at"),
        )
        
        # Get context for segment count and duration
        context = await self._get_context_async(node.id, depth=1)
        segment_count = 0
        total_duration = 0.0
        if context is not None:
            segments = self._get_segment_nodes(context)
            segment_count = len(segments)
            timing = self._compute_timing_stats(segments)
            total_duration = timing['total_duration']
        
        summaries.append(DocumentSummary(
            document_id=node.id,
            title=node.properties.get('title', 'Untitled'),
            media_type=node.properties.get('media_type', 'unknown'),
            segment_count=segment_count,
            total_duration=total_duration,
            created_at=node.created_at or 0.0,
        ))
    
    # Sort by created_at descending (newest first)
    summaries.sort(key=lambda s: s.created_at, reverse=True)
    
    if DEBUG_MANAGEMENT_SERVICE:
        print(f"[ManagementService] Returning {len(summaries)} summaries")
    
    return summaries

@patch
def list_documents(self:ManagementService) -> List[DocumentSummary]:  # All documents sorted newest first
    """List all documents with summary info synchronously."""
    return asyncio.get_event_loop().run_until_complete(
        self.list_documents_async()
    )

### Get Document Detail

In [ ]:
#| export
@patch
async def get_document_detail_async(
    self:ManagementService,
    document_id: str,  # UUID of the Document node
) -> Optional[DocumentDetail]:  # Full detail or None if not found
    """Get full document detail with integrity checks and samples."""
    if DEBUG_MANAGEMENT_SERVICE:
        print(f"[ManagementService] get_document_detail called with id: {document_id}")
    
    if not self.is_available():
        return None
    
    # depth=2 gets Document + Segments + NEXT edges between segments
    context = await self._get_context_async(document_id, depth=2)
    if context is None or not context.nodes:
        if DEBUG_MANAGEMENT_SERVICE:
            print(f"[ManagementService] No context found for {document_id}")
        return None
    
    # Extract Document node
    document_node = self._find_document_node(context, document_id)
    if document_node is None:
        if DEBUG_MANAGEMENT_SERVICE:
            print(f"[ManagementService] Document node not found in context")
        return None
    
    # Extract and sort Segment nodes
    segment_nodes = self._get_segment_nodes(context)
    segment_count = len(segment_nodes)
    
    if DEBUG_MANAGEMENT_SERVICE:
        print(f"[ManagementService] Found {segment_count} segments, {len(context.edges)} edges")
    
    # Compute stats
    edge_counts = self._count_edges_by_type(context.edges)
    timing_stats = self._compute_timing_stats(segment_nodes)
    source_stats = self._compute_source_stats(segment_nodes)
    
    # Extract sample segments
    first_samples = self._extract_samples(segment_nodes[:3])
    last_samples = self._extract_samples(segment_nodes[-3:]) if segment_count > 3 else []
    
    # Compute integrity flags
    has_starts_with = edge_counts['STARTS_WITH'] >= 1
    next_chain_complete = edge_counts['NEXT'] == max(0, segment_count - 1)
    part_of_complete = edge_counts['PART_OF'] == segment_count
    all_have_timing = timing_stats['missing_count'] == 0
    all_have_sources = source_stats['missing_count'] == 0
    all_checks_passed = (
        has_starts_with and next_chain_complete and part_of_complete
        and all_have_timing and all_have_sources
    )
    
    return DocumentDetail(
        document_id=document_id,
        title=document_node.properties.get('title', 'Untitled'),
        media_type=document_node.properties.get('media_type', 'unknown'),
        created_at=document_node.created_at or 0.0,
        updated_at=document_node.updated_at or 0.0,
        segment_count=segment_count,
        total_duration=timing_stats['total_duration'],
        avg_segment_duration=timing_stats['avg_duration'],
        has_starts_with=has_starts_with,
        next_chain_complete=next_chain_complete,
        next_count=edge_counts['NEXT'],
        part_of_complete=part_of_complete,
        part_of_count=edge_counts['PART_OF'],
        all_have_timing=all_have_timing,
        segments_missing_timing=timing_stats['missing_count'],
        all_have_sources=all_have_sources,
        segments_missing_sources=source_stats['missing_count'],
        all_checks_passed=all_checks_passed,
        source_plugins=source_stats['plugins'],
        first_segments=first_samples,
        last_segments=last_samples,
    )

@patch
def get_document_detail(
    self:ManagementService,
    document_id: str,  # UUID of the Document node
) -> Optional[DocumentDetail]:  # Full detail or None if not found
    """Get full document detail with integrity checks and samples synchronously."""
    return asyncio.get_event_loop().run_until_complete(
        self.get_document_detail_async(document_id)
    )

### Delete Documents

In [ ]:
#| export
@patch
async def delete_document_async(
    self:ManagementService,
    document_id: str,  # UUID of the Document node to delete
) -> bool:  # True if deletion succeeded
    """Delete a single document and all its segments via cascade."""
    if DEBUG_MANAGEMENT_SERVICE:
        print(f"[ManagementService] delete_document called with id: {document_id}")
    
    if not self.is_available():
        return False
    
    result = await self._delete_nodes_async([document_id], cascade=True)
    if result is None:
        return False
    
    deleted = result.get("deleted_count", 0)
    if DEBUG_MANAGEMENT_SERVICE:
        print(f"[ManagementService] Deleted {deleted} nodes for document {document_id}")
    
    return deleted > 0

@patch
def delete_document(
    self:ManagementService,
    document_id: str,  # UUID of the Document node to delete
) -> bool:  # True if deletion succeeded
    """Delete a single document and all its segments synchronously."""
    return asyncio.get_event_loop().run_until_complete(
        self.delete_document_async(document_id)
    )

@patch
async def delete_documents_async(
    self:ManagementService,
    document_ids: List[str],  # UUIDs of Document nodes to delete
) -> int:  # Number of documents successfully deleted
    """Delete multiple documents and all their segments via cascade."""
    if DEBUG_MANAGEMENT_SERVICE:
        print(f"[ManagementService] delete_documents called with {len(document_ids)} ids")
    
    if not self.is_available() or not document_ids:
        return 0
    
    result = await self._delete_nodes_async(document_ids, cascade=True)
    if result is None:
        return 0
    
    deleted = result.get("deleted_count", 0)
    if DEBUG_MANAGEMENT_SERVICE:
        print(f"[ManagementService] Bulk delete removed {deleted} nodes")
    
    return deleted

@patch
def delete_documents(
    self:ManagementService,
    document_ids: List[str],  # UUIDs of Document nodes to delete
) -> int:  # Number of documents successfully deleted
    """Delete multiple documents and all their segments synchronously."""
    return asyncio.get_event_loop().run_until_complete(
        self.delete_documents_async(document_ids)
    )

### Export

In [ ]:
#| export
@patch
async def export_document_async(
    self:ManagementService,
    document_id: str,  # UUID of the Document node to export
) -> Optional[ExportBundle]:  # Export bundle or None if not found
    """Export a single document's subgraph as an ExportBundle."""
    if DEBUG_MANAGEMENT_SERVICE:
        print(f"[ManagementService] export_document called with id: {document_id}")
    
    if not self.is_available():
        return None
    
    context = await self._get_context_async(document_id, depth=2)
    if context is None or not context.nodes:
        return None
    
    return self._context_to_bundle(context, document_count=1)

@patch
def export_document(
    self:ManagementService,
    document_id: str,  # UUID of the Document node to export
) -> Optional[ExportBundle]:  # Export bundle or None if not found
    """Export a single document's subgraph synchronously."""
    return asyncio.get_event_loop().run_until_complete(
        self.export_document_async(document_id)
    )

@patch
async def export_all_async(self:ManagementService) -> Optional[ExportBundle]:  # Export bundle or None if error
    """Export the entire graph database as an ExportBundle."""
    if DEBUG_MANAGEMENT_SERVICE:
        print("[ManagementService] export_all called")
    
    if not self.is_available():
        return None
    
    context = await self._export_graph_async()
    if context is None:
        return None
    
    # Count Document nodes in the exported graph
    doc_count = sum(1 for n in context.nodes if n.label == 'Document')
    
    if DEBUG_MANAGEMENT_SERVICE:
        print(f"[ManagementService] Exporting {doc_count} documents, "
              f"{len(context.nodes)} nodes, {len(context.edges)} edges")
    
    return self._context_to_bundle(context, document_count=doc_count)

@patch
def export_all(self:ManagementService) -> Optional[ExportBundle]:  # Export bundle or None if error
    """Export the entire graph database synchronously."""
    return asyncio.get_event_loop().run_until_complete(
        self.export_all_async()
    )

### Import

In [ ]:
#| export
@patch
async def import_graph_async(
    self:ManagementService,
    bundle_data: Dict[str, Any],  # Parsed JSON from export file
    merge_strategy: str = "skip",  # skip, overwrite, or merge
) -> ImportResult:  # Result with counts and any errors
    """Validate and import graph data from an export bundle."""
    if DEBUG_MANAGEMENT_SERVICE:
        print(f"[ManagementService] import_graph called with strategy: {merge_strategy}")
    
    # Validate format
    fmt = bundle_data.get("format")
    if fmt != self.EXPORT_FORMAT:
        return ImportResult(
            success=False, nodes_created=0, edges_created=0,
            nodes_skipped=0, edges_skipped=0,
            errors=[f"Invalid format: expected '{self.EXPORT_FORMAT}', got '{fmt}'"],
        )
    
    # Validate version
    version = bundle_data.get("version", "")
    if not version.startswith("1."):
        return ImportResult(
            success=False, nodes_created=0, edges_created=0,
            nodes_skipped=0, edges_skipped=0,
            errors=[f"Unsupported version: '{version}' (expected 1.x)"],
        )
    
    # Extract graph data
    graph_data = bundle_data.get("graph")
    if graph_data is None or not isinstance(graph_data, dict):
        return ImportResult(
            success=False, nodes_created=0, edges_created=0,
            nodes_skipped=0, edges_skipped=0,
            errors=["Missing or invalid 'graph' field"],
        )
    
    if not self.is_available():
        return ImportResult(
            success=False, nodes_created=0, edges_created=0,
            nodes_skipped=0, edges_skipped=0,
            errors=["Graph plugin not available"],
        )
    
    # Call plugin import
    result = await self._import_graph_async(graph_data, merge_strategy)
    if result is None:
        return ImportResult(
            success=False, nodes_created=0, edges_created=0,
            nodes_skipped=0, edges_skipped=0,
            errors=["Plugin import_graph returned no result"],
        )
    
    nodes_created = result.get("nodes_created", 0)
    edges_created = result.get("edges_created", 0)
    nodes_skipped = result.get("nodes_skipped", 0)
    edges_skipped = result.get("edges_skipped", 0)
    
    if DEBUG_MANAGEMENT_SERVICE:
        print(f"[ManagementService] Import result: {nodes_created} nodes, "
              f"{edges_created} edges created, {nodes_skipped} nodes skipped")
    
    return ImportResult(
        success=True,
        nodes_created=nodes_created,
        edges_created=edges_created,
        nodes_skipped=nodes_skipped,
        edges_skipped=edges_skipped,
    )

@patch
def import_graph(
    self:ManagementService,
    bundle_data: Dict[str, Any],  # Parsed JSON from export file
    merge_strategy: str = "skip",  # skip, overwrite, or merge
) -> ImportResult:  # Result with counts and any errors
    """Validate and import graph data synchronously."""
    return asyncio.get_event_loop().run_until_complete(
        self.import_graph_async(bundle_data, merge_strategy)
    )

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()